# Equal-Weight S&P 500 Index Fund

**Author: Abraham Sobowale**

This notebook builds a Python tool that takes a portfolio value and returns the number of shares of each S&P 500 constituent to buy for an *equal-weight* version of the index (every company weighted identically, rather than by market cap).

I built this from the FreeCodeCamp *Algorithmic Trading in Python* template and reworked it into my own implementation, using `pandas` for data handling and the IEX Cloud API (sandbox) for pricing.

## Library Imports

In [1]:
import numpy as np
import requests
import pandas as pd
import math
import xlsxwriter

## Importing Our List of Stocks

Next I import the constituents of the S&P 500.

These constituents change over time, so in an ideal world you would connect directly to the index provider (Standard & Poor's) and pull their real-time constituents on a regular basis.

Paying for access to the index provider's API is outside the scope of this project. 

There's a static version of the S&P 500 constituents available here. [Click this link to download them now](https://drive.google.com/file/d/1ZJSpbY69DVckVZlO9cC6KkgfSufybcHN/view?usp=sharing). Move this file into the `starter-files` folder so it can be accessed by other files in that directory.

Now it's time to import these stocks to our Jupyter Notebook file.

In [2]:
stocks = pd.read_csv("sp_500_stocks.csv")
stocks

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
...,...
500,YUM
501,ZBH
502,ZBRA
503,ZION


In [3]:
#stocks[0]

KeyError: 0

## Acquiring an API Token

Market data comes from the IEX Cloud API. I keep the token in a local `sec.py` file (`IEX_CLOUD_API_TOKEN = '...'`) that is git-ignored so it never gets committed, and import it with `from sec import IEX_CLOUD_API_TOKEN`. The token used here is a **sandbox** token, so the data is randomly generated and free to call.

In [ ]:
from sec import IEX_CLOUD_API_TOKEN

## Making Our First API Call

Now it's time to structure our API calls to IEX cloud. 

We need the following information from the API:

* Market capitalization for each stock
* Price of each stock



In [3]:
symbol = "AAPL"
# COMPANY PROFILE DATA API
api_url = f"https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey=UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
data = requests.get(api_url).json()
print(data)

[{'symbol': 'AAPL', 'price': 227.8703, 'marketCap': 3381686400120, 'beta': 1.165, 'lastDividend': 1.27, 'range': '169.21-260.1', 'change': -1.4797, 'changePercentage': -0.64517, 'volume': 41034348, 'averageVolume': 56098309, 'companyName': 'Apple Inc.', 'currency': 'USD', 'cik': '0000320193', 'isin': 'US0378331005', 'cusip': '037833100', 'exchangeFullName': 'NASDAQ Global Select', 'exchange': 'NASDAQ', 'industry': 'Consumer Electronics', 'website': 'https://www.apple.com', 'description': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple TV, Apple Watch, Beats products, and HomePod. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and

In [4]:
# turn list -> dict
data = data[0]

## Parsing Our API Call

The API call that I executed in the last code block contains all of the information required to build our equal-weight S&P 500 strategy. 

With that said, the data isn't in a proper format yet. We need to parse it first.

In [5]:
prices = data["price"]
market_cap = data["marketCap"]

## Adding Our Stocks Data to a Pandas DataFrame

The next thing I need to do is add our stock's price and market capitalization to a pandas DataFrame. Think of a DataFrame like the Python version of a spreadsheet. It stores tabular data.

In [6]:
# 1. First ensure final_df is created properly
my_columns = ["Ticker", "Stock Price", "Market Capitalization", "Number of Shares to Buy"]
final_df = pd.DataFrame(columns=my_columns)

# 2. Verify it's a DataFrame before concat
print(type(final_df))  # Should show <class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>


In [8]:
all_data = []

# Append dictionaries to list
all_data.append({
    "Ticker": symbol,
    "Stock Price": prices,
    "Market Capitalization": market_cap,
    "Number of Shares to Buy": "N/A"
})

# Convert to DataFrame once
final_df = pd.DataFrame(all_data, columns=my_columns)
print(final_df)

  Ticker  Stock Price  Market Capitalization Number of Shares to Buy
0   AAPL       212.01          3166538958000                     N/A


## Looping Through The Tickers in Our List of Stocks

Using the same logic that I outlined above, I can pull data for all S&P 500 stocks and store their data in the DataFrame using a `for` loop.

In [ ]:
# single api requests (SLOW)
rows = []
for stock in stocks["Ticker"][:10]:
    api_url = f"https://financialmodelingprep.com/stable/profile?symbol={stock}&apikey=UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
    data = requests.get(api_url).json()
    data = data[0]
    rows.append({
        my_columns[0]: stock,
        my_columns[1]: data["price"],
        my_columns[2]: data["marketCap"],
        my_columns[3]: "N/A"
    })
final_df = pd.DataFrame(rows, columns=my_columns)

In [ ]:
final_df

## Using Batch API Calls to Improve Performance

Batch API calls are one of the easiest ways to improve the performance of your code.

This is because HTTP requests are typically one of the slowest components of a script.

Also, API providers will often give you discounted rates for using batch API calls since they are easier for the API provider to respond to.

IEX Cloud limits their batch API calls to 100 tickers per request. Still, this reduces the number of API calls I'll make in this section from 500 to 5 - huge improvement! In this section, I'll split our list of stocks into groups of 100 and then make a batch API call for each group.

In [7]:
#  generator that yields evenly-sized chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [8]:
# Spliting based on the ticker limit of API
symbol_groups = list(chunks(stocks["Ticker"], 100))
symbol_str = []
for i in range(len(symbol_groups)):
    symbol_str.append(",".join(symbol_groups[i]))

final_df = pd.DataFrame(columns=my_columns)
rows = []

for symbol_s in symbol_str:
    batch_api_call_url = f"https://financialmodelingprep.com/api/v3/quote/{symbol_s}?apikey=UTyHVduohyBpg6IFqApjemQi7MtN1dTB"
    data = requests.get(batch_api_call_url).json()
    print(data)
    for stock_data in data:  # Loop through each stock's data in the batch response
        symbol = stock_data["symbol"]  # Get the stock symbol from the response
        rows.append({
            my_columns[0]: symbol,
            my_columns[1]: stock_data["price"],
            my_columns[2]: stock_data.get("marketCap", "N/A"),  # Use .get() to handle missing keys
            my_columns[3]: "N/A"  # Or any default value you want
        })
    
final_df = pd.DataFrame(rows, columns=my_columns) 

final_df

[{'symbol': 'A', 'name': 'Agilent Technologies, Inc.', 'price': 114.59, 'changesPercentage': -0.02617344, 'change': -0.03, 'dayLow': 113.73, 'dayHigh': 115.54, 'yearHigh': 153.84, 'yearLow': 96.43, 'marketCap': 32551008350, 'priceAvg50': 117.1134, 'priceAvg200': 125.0594, 'exchange': 'NYSE', 'volume': 547196, 'avgVolume': 1979854, 'open': 114.96, 'previousClose': 114.62, 'eps': 4.06, 'pe': 28.22, 'earningsAnnouncement': '2025-08-27T20:00:00.000+0000', 'sharesOutstanding': 284065000, 'timestamp': 1754937049}, {'symbol': 'AAL', 'name': 'American Airlines Group Inc.', 'price': 11.615, 'changesPercentage': -0.04302926, 'change': -0.005, 'dayLow': 11.56, 'dayHigh': 11.76, 'yearHigh': 19.1, 'yearLow': 8.5, 'marketCap': 7663913835, 'priceAvg50': 11.552, 'priceAvg200': 13.26855, 'exchange': 'NASDAQ', 'volume': 37918036, 'avgVolume': 65670219.1, 'open': 11.61, 'previousClose': 11.62, 'eps': 0.84, 'pe': 13.83, 'earningsAnnouncement': '2025-10-23T04:00:00.000+0000', 'sharesOutstanding': 659829000

,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,114.5900,32551008350,N/A
1,AAL,11.6150,7663913835,N/A
2,AAP,59.3400,3555985104,N/A
3,AAPL,227.9388,3382702967520,N/A
4,ABBV,197.7750,349381404000,N/A
...,...,...,...,...
489,YUM,141.6250,39306036000,N/A
490,ZBH,99.8000,19769980800,N/A
491,ZBRA,311.8800,15857569788,N/A
492,ZION,51.9933,7675458919,N/A


In [12]:
final_df

,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,119.6500,33988377250,N/A
1,AAL,11.5950,7650717255,N/A
2,AAP,56.3900,3379204584,N/A
3,AAPL,212.0899,3167732328420,N/A
4,ABBV,191.6700,338565888000,N/A
...,...,...,...,...
489,YUM,144.6325,40202772863,N/A
490,ZBH,95.3800,18870742240,N/A
491,ZBRA,332.9200,16930413556,N/A
492,ZION,54.9850,8114191435,N/A


## Calculating the Number of Shares to Buy

As you can see in the DataFrame above, I stil haven't calculated the number of shares of each stock to buy.

We'll do that next.

In [ ]:
portfolio_size = input("What is your portfolio size? ")

while True:
    try:
        val = float(portfolio_size)
        print(portfolio_size)
        break  # Exit the loop if conversion succeeds
    except ValueError:
        print("Please enter a valid number.")
        portfolio_size = input("What is your portfolio size? ")

In [ ]:
position_size = val/len(final_df.index)
for i in range(len(final_df.index)):    
    final_df["Number of Shares to Buy"] = (position_size / final_df["Stock Price"]).apply(math.floor)
final_df

## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

### Initializing our XlsxWriter Object

In [ ]:
writer = pd.ExcelWriter("recommended trades.xlsx", engine = "xlsxwriter")
final_df.to_excel(writer, "Recommended Trades", index = False)

### Creating the Formats We'll Need For Our `.xlsx` File

Formats include colors, fonts, and also symbols like `%` and `$`. We'll need four main formats for our Excel document:
* String format for tickers
* \\$XX.XX format for stock prices
* \\$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

In [ ]:
#setting colours
backgroud_col = "#000000"
font_col = "#D271DE"

string_format = writer.book.add_format(
    {
        "font_color": font_col,
        "bg_color": backgroud_col,
        "border": 1
    }
)

dollar_format = writer.book.add_format(
    {
        "num_format": "£0.00", # GDP format
        "font_color": font_col,
        "bg_color": backgroud_col,
        "border": 1
    }
)

integer_format = writer.book.add_format(
    {
        "num_format": "0",
        "font_color": font_col,
        "bg_color": backgroud_col,
        "border": 1
    }
)

### Applying the Formats to the Columns of Our `.xlsx` File

We can use the `set_column` method applied to the `writer.sheets['Recommended Trades']` object to apply formats to specific columns of our spreadsheets.

Here's an example:

```python
writer.sheets['Recommended Trades'].set_column('B:B', #This tells the method to apply the format to column B
                     18, #This tells the method to apply a column width of 18 pixels
                     string_template #This applies the format 'string_template' to the column
                    )
```

In [ ]:
#writer.sheets["Recommended Trades"].set_column("A:A", 18, string_format )
#writer.sheets["Recommended Trades"].set_column("B:B", 18, string_format )
#writer.sheets["Recommended Trades"].set_column("C:C", 18, string_format )
#writer.sheets["Recommended Trades"].set_column("D:D", 18, string_format )
#writer.close()

# This is to color the headings
writer.sheets["Recommended Trades"].set_column("A:A", 18, string_format )
writer.sheets["Recommended Trades"].set_column("B:B", 18, string_format )
writer.sheets["Recommended Trades"].set_column("C:C", 18, string_format )
writer.sheets["Recommended Trades"].set_column("D:D", 18, string_format )

This code works, but it violates the software principle of "Don't Repeat Yourself". 

Let's simplify this by putting it in 2 loops:

In [ ]:
columns_format = {
    "A": ["Ticker", string_format],
    "B": ["Stock Price", dollar_format],
    "C": ["Market Capitalization", dollar_format],
    "D": ["Number of Shares to Buy", integer_format]
}
for columns in columns_format.keys():
    writer.sheets["Recommended Trades"].set_column(f"{columns}:{columns}", 18, columns_format[columns][1] )

## Saving Our Excel Output

Saving our Excel file is very easy:

In [ ]:
writer.close()